# Notebook 04 — Safety Classifier Dataset
**Project:** SRH AI Platform — Rwandan Youth & PWDs  
**Purpose:** Download, clean, and merge labeled SAFE/UNSAFE datasets into a single training-ready CSV for Model 1 (Safety Classifier).

## What this notebook produces
| Output file | Records (target) | Used by |
|---|---|---|
| `safety_classifier_raw.csv` | ~50k pre-filter rows | Inspection |
| `safety_classifier_clean.csv` | ~10k–15k balanced rows | Training |
| `safety_classifier_final.csv` | 2k–5k stratified sample | Model 1 training |

## Sources used
| # | Dataset | Label type | Rows | Role |
|---|---|---|---|---|
| 1 | BeaverTails (PKU-Alignment) | `is_safe` boolean | 301k train | UNSAFE class (primary) |
| 2 | ToxicChat 0124 (LMSYS) | `toxicity` 0/1 | ~10k | UNSAFE + SAFE (real chatbot) |
| 3 | Anthropic HH-RLHF harmless-base | chosen/rejected pairs | ~42k | SAFE class (gold standard) |
| 4 | NVIDIA AEGIS Safety Dataset 1.0 | `label` safe/unsafe | ~26k | UNSAFE supplement |

## Label schema (output)
```
label = 0  →  SAFE   (legitimate SRH or health query)
label = 1  →  UNSAFE (harmful, exploitative, or abusive query)
```

---
## Cell 1 — Install dependencies

In [ ]:
# Run this cell first in Colab
!pip install -q datasets pandas scikit-learn matplotlib seaborn tqdm
print("Dependencies installed.")

---
## Cell 2 — Imports and config

In [ ]:
import re
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

# ── Config ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = 'data/safety'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Target samples PER CLASS in the final balanced dataset
# Increase to 5000 if you want a larger training set
TARGET_PER_CLASS = 2500

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Target per class: {TARGET_PER_CLASS} (total ~{TARGET_PER_CLASS * 2})")

---
## Cell 3 — Download Source 1: BeaverTails

**What it is:** 301k+ human-labeled Q&A pairs from PKU-Alignment.  
**Why we use it:** Each row has `is_safe` (True/False). The UNSAFE rows cover 14 harm categories — sexual violence, coercion, self-harm, exploitation, child abuse — exactly what your SRH chatbot safety filter must block.  
**What we extract:** The `prompt` column + `is_safe` label. We keep both safe and unsafe rows.

In [ ]:
print("Downloading BeaverTails 30k_train split...")
bt_raw = load_dataset("PKU-Alignment/BeaverTails", split="30k_train")
bt_df = bt_raw.to_pandas()

print(f"\nRaw shape: {bt_df.shape}")
print(f"Columns: {list(bt_df.columns)}")
print(f"\nSample row:")
print(bt_df.iloc[0])

In [ ]:
# ── Inspect label structure ───────────────────────────────────────────────────
# BeaverTails has response_0 and response_1 columns
# is_response_0_safe and is_response_1_safe are the label columns
# We extract the PROMPT as the text + use is_response_0_safe as proxy for query safety

print("Label columns available:")
label_cols = [c for c in bt_df.columns if 'safe' in c.lower() or 'categor' in c.lower()]
print(label_cols)

print("\nValue counts for is_response_0_safe:")
if 'is_response_0_safe' in bt_df.columns:
    print(bt_df['is_response_0_safe'].value_counts())
elif 'is_safe' in bt_df.columns:
    print(bt_df['is_safe'].value_counts())

print("\nHarm category sample:")
cat_col = [c for c in bt_df.columns if 'categor' in c.lower()]
if cat_col:
    print(bt_df[cat_col[0]].value_counts().head(10))

In [ ]:
# ── Extract text + label ──────────────────────────────────────────────────────
# Prompt column is the user query — this is what the safety classifier will see at inference
# An UNSAFE query = one where the expected response is unsafe

bt_label_col = 'is_response_0_safe' if 'is_response_0_safe' in bt_df.columns else 'is_safe'
bt_text_col  = 'prompt'             if 'prompt' in bt_df.columns else bt_df.columns[0]

bt_extracted = bt_df[[bt_text_col, bt_label_col]].copy()
bt_extracted.columns = ['text', 'is_safe_raw']

# Convert to our binary label: 0=SAFE, 1=UNSAFE
bt_extracted['label']  = bt_extracted['is_safe_raw'].apply(lambda x: 0 if x else 1)
bt_extracted['source'] = 'beavertails'
bt_extracted = bt_extracted[['text', 'label', 'source']]

print(f"Extracted shape: {bt_extracted.shape}")
print(f"Label distribution:")
print(bt_extracted['label'].value_counts())
print(f"\nSample UNSAFE (label=1):")
print(bt_extracted[bt_extracted['label']==1].iloc[0]['text'][:300])
print(f"\nSample SAFE (label=0):")
print(bt_extracted[bt_extracted['label']==0].iloc[0]['text'][:300])

---
## Cell 4 — Download Source 2: ToxicChat (LMSYS)

**What it is:** 10k real user–AI chatbot conversations from Vicuna/Chatbot Arena, human-annotated for toxicity and jailbreak attempts.  
**Why we use it:** Unlike BeaverTails (which uses clean prompts), ToxicChat contains real user phrasing — including subtle jailbreaks and indirect harmful requests. This makes your safety classifier more robust against real-world inputs.  
**What we extract:** `user_input` + `toxicity` (0=safe, 1=toxic).

In [ ]:
print("Downloading ToxicChat 0124...")
tc_raw = load_dataset("lmsys/toxic-chat", "toxicchat0124", split="train")
tc_df = tc_raw.to_pandas()

print(f"\nRaw shape: {tc_df.shape}")
print(f"Columns: {list(tc_df.columns)}")
print(f"\nLabel distribution:")
print(tc_df['toxicity'].value_counts())
print(f"\nJailbreak label distribution:")
if 'jailbreaking' in tc_df.columns:
    print(tc_df['jailbreaking'].value_counts())

In [ ]:
# ── Extract text + label ──────────────────────────────────────────────────────
tc_text_col  = 'user_input'  if 'user_input' in tc_df.columns else 'human_turn'
tc_label_col = 'toxicity'

tc_extracted = tc_df[[tc_text_col, tc_label_col]].copy()
tc_extracted.columns = ['text', 'label']

# toxicity is already 0=safe, 1=toxic — matches our schema
tc_extracted['label']  = tc_extracted['label'].astype(int)
tc_extracted['source'] = 'toxicchat'
tc_extracted = tc_extracted[['text', 'label', 'source']]

print(f"Extracted shape: {tc_extracted.shape}")
print(f"Label distribution:")
print(tc_extracted['label'].value_counts())
print(f"\nSample UNSAFE (label=1):")
print(tc_extracted[tc_extracted['label']==1].iloc[0]['text'][:300])

---
## Cell 5 — Download Source 3: Anthropic HH-RLHF (harmless-base)

**What it is:** ~42k human preference pairs annotated for harmlessness. Each row has a `chosen` and `rejected` response to the same prompt.  
**Why we use it:** The `chosen` responses are the gold standard for SAFE answers — human reviewers explicitly selected them as the more harmless option. The prompts from the `rejected` side tend to be toward harmful territory.  
**What we extract:** We pull the final human turn from each conversation. Chosen conversations → SAFE (0). Rejected conversations → UNSAFE (1).

In [ ]:
print("Downloading Anthropic HH-RLHF harmless-base...")
hh_raw = load_dataset("Anthropic/hh-rlhf", data_dir="harmless-base", split="train")
hh_df = hh_raw.to_pandas()

print(f"\nRaw shape: {hh_df.shape}")
print(f"Columns: {list(hh_df.columns)}")
print(f"\nSample chosen (first 400 chars):")
print(hh_df['chosen'].iloc[0][:400])

In [ ]:
# ── Extract the LAST Human turn from each conversation ───────────────────────
# Format: "Human: <query>\n\nAssistant: <response>\n\nHuman: ..."
# We extract the last Human: turn as the "query" text for classification

def extract_last_human_turn(text):
    """Extract the final Human: turn from an HH-RLHF conversation string."""
    if not isinstance(text, str):
        return None
    # Split on Human: / Assistant: markers
    turns = re.split(r'\n\nHuman:|\n\nAssistant:', text)
    human_turns = []
    # First token is always empty or first human turn
    for i, turn in enumerate(turns):
        # Odd-indexed after first split: human turns (since format is Human/Asst alternating)
        t = turn.strip()
        if t and i % 2 == 1:  # Human turns are odd-indexed in the split
            human_turns.append(t)
    if human_turns:
        return human_turns[-1]  # Last human query
    # Fallback: just return the first 300 chars
    return text[:300].strip()

tqdm.pandas(desc="Extracting chosen human turns")
hh_df['text_chosen']   = hh_df['chosen'].progress_apply(extract_last_human_turn)
tqdm.pandas(desc="Extracting rejected human turns")
hh_df['text_rejected'] = hh_df['rejected'].progress_apply(extract_last_human_turn)

print(f"Sample extracted chosen turn:")
print(repr(hh_df['text_chosen'].iloc[5]))
print(f"\nSample extracted rejected turn:")
print(repr(hh_df['text_rejected'].iloc[5]))

In [ ]:
# ── Build SAFE (chosen) and UNSAFE (rejected) rows ───────────────────────────
hh_safe = pd.DataFrame({
    'text':   hh_df['text_chosen'].dropna(),
    'label':  0,
    'source': 'hh_rlhf_chosen'
})

hh_unsafe = pd.DataFrame({
    'text':   hh_df['text_rejected'].dropna(),
    'label':  1,
    'source': 'hh_rlhf_rejected'
})

hh_extracted = pd.concat([hh_safe, hh_unsafe], ignore_index=True)

print(f"HH-RLHF extracted shape: {hh_extracted.shape}")
print(f"Label distribution:")
print(hh_extracted['label'].value_counts())

---
## Cell 6 — Download Source 4: NVIDIA AEGIS Safety Dataset

**What it is:** 26.5k human–LLM interaction samples annotated across a safety risk taxonomy.  
**Why we use it:** It adds diverse phrasing diversity to the UNSAFE class — different from BeaverTails' phrasing style. Built from real user prompts + Mistral responses.  
**What we extract:** `text` (the prompt) + `label` column.

In [ ]:
print("Downloading NVIDIA AEGIS Safety Dataset 1.0...")
aegis_raw = load_dataset("nvidia/Aegis-AI-Content-Safety-Dataset-1.0", split="train")
aegis_df = aegis_raw.to_pandas()

print(f"\nRaw shape: {aegis_df.shape}")
print(f"Columns: {list(aegis_df.columns)}")
print(f"\nLabel distribution:")
print(aegis_df[aegis_df.columns[-1]].value_counts().head(15))

In [ ]:
# ── Inspect and map the label column ─────────────────────────────────────────
# AEGIS uses text labels like 'safe', 'unsafe', category names
# Find the correct label column
print("All columns and first unique values:")
for col in aegis_df.columns:
    unique_vals = aegis_df[col].dropna().unique()[:5]
    print(f"  {col}: {unique_vals}")

In [ ]:
# ── Identify text and label columns, map to 0/1 ──────────────────────────────
# Typical AEGIS structure: 'text' or 'prompt' column + 'label_0' or 'labels' column

aegis_text_col = None
aegis_label_col = None

for col in aegis_df.columns:
    if col.lower() in ['text', 'prompt', 'instruction', 'query', 'content']:
        aegis_text_col = col
    if 'label' in col.lower() or 'safe' in col.lower() or 'class' in col.lower():
        aegis_label_col = col

# Fallback: first string col = text, last col = label
if aegis_text_col is None:
    aegis_text_col = aegis_df.select_dtypes(include='object').columns[0]
if aegis_label_col is None:
    aegis_label_col = aegis_df.columns[-1]

print(f"Using text col: '{aegis_text_col}'")
print(f"Using label col: '{aegis_label_col}'")
print(f"Label unique values: {aegis_df[aegis_label_col].unique()[:10]}")

# Map safe→0, everything else→1
def map_aegis_label(val):
    if isinstance(val, str) and val.strip().lower() == 'safe':
        return 0
    return 1

aegis_extracted = pd.DataFrame({
    'text':   aegis_df[aegis_text_col],
    'label':  aegis_df[aegis_label_col].apply(map_aegis_label),
    'source': 'aegis'
})

print(f"\nExtracted shape: {aegis_extracted.shape}")
print(f"Label distribution:")
print(aegis_extracted['label'].value_counts())

---
## Cell 7 — Raw merge: combine all 4 sources

In [ ]:
# ── Merge all sources ─────────────────────────────────────────────────────────
all_sources = [
    ('BeaverTails',  bt_extracted),
    ('ToxicChat',    tc_extracted),
    ('HH-RLHF',      hh_extracted),
    ('AEGIS',        aegis_extracted),
]

frames = []
for name, df in all_sources:
    safe_n   = (df['label'] == 0).sum()
    unsafe_n = (df['label'] == 1).sum()
    total    = len(df)
    print(f"{name:15s} → SAFE={safe_n:6,} | UNSAFE={unsafe_n:6,} | TOTAL={total:7,}")
    frames.append(df)

raw_merged = pd.concat(frames, ignore_index=True)

print(f"\n{'='*60}")
print(f"COMBINED RAW → {len(raw_merged):,} rows")
print(f"SAFE   (0): {(raw_merged['label']==0).sum():,}")
print(f"UNSAFE (1): {(raw_merged['label']==1).sum():,}")

# Save raw merged
raw_merged.to_csv(f'{OUTPUT_DIR}/safety_classifier_raw.csv', index=False)
print(f"\nSaved: {OUTPUT_DIR}/safety_classifier_raw.csv")

---
## Cell 8 — Cleaning Step 1: Drop nulls and empty strings

**Why:** Null or empty text rows will crash tokenizers and padding logic downstream.

In [ ]:
df = raw_merged.copy()

before = len(df)

# 1. Drop nulls
df = df.dropna(subset=['text', 'label'])
print(f"After dropna:          {len(df):,} rows (removed {before - len(df):,})")

# 2. Drop empty / whitespace-only strings
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 0]
print(f"After empty-str drop:  {len(df):,} rows")

# 3. Ensure label is integer 0 or 1 only (drop any rows with unexpected values)
df['label'] = pd.to_numeric(df['label'], errors='coerce').astype('Int64')
df = df[df['label'].isin([0, 1])]
df['label'] = df['label'].astype(int)
print(f"After label validation: {len(df):,} rows")
print(f"Label distribution:")
print(df['label'].value_counts())

---
## Cell 9 — Cleaning Step 2: Remove duplicates

**Why:** Duplicate rows cause the model to overfit to repeated examples, inflating evaluation metrics without improving generalization.

In [ ]:
before = len(df)

# Exact duplicates on text (case-sensitive)
df = df.drop_duplicates(subset='text', keep='first')
print(f"After exact dedup:    {len(df):,} rows (removed {before - len(df):,} duplicates)")

# Near-duplicates: normalise text (lowercase + collapse whitespace) then dedup
df['text_norm'] = df['text'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
before2 = len(df)
df = df.drop_duplicates(subset='text_norm', keep='first')
df = df.drop(columns='text_norm')
print(f"After near-dedup:     {len(df):,} rows (removed {before2 - len(df):,} near-duplicates)")

---
## Cell 10 — Cleaning Step 3: Length filter

**Why:** Very short texts (< 5 words) lack context for classification. Very long texts (> 512 tokens) will be truncated by most models anyway — using them as-is wastes memory. We keep a practical middle range.

In [ ]:
# Word-count based filter (faster than tokenizer; good proxy)
df['word_count'] = df['text'].str.split().str.len()

print("Word count distribution before filter:")
print(df['word_count'].describe())

fig, ax = plt.subplots(figsize=(10, 3))
df['word_count'].clip(0, 200).hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(5,   color='red',   linestyle='--', label='Min (5 words)')
ax.axvline(150, color='orange',linestyle='--', label='Max (150 words)')
ax.set_title('Word count distribution (clipped at 200)')
ax.set_xlabel('Words')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_word_count.png', dpi=120)
plt.show()

before = len(df)
# Keep 5–150 words: enough context, not exceeding standard tokenizer limits
df = df[(df['word_count'] >= 5) & (df['word_count'] <= 150)]
df = df.drop(columns='word_count')
print(f"\nAfter length filter (5–150 words): {len(df):,} rows (removed {before - len(df):,})")
print(f"Label distribution:")
print(df['label'].value_counts())

---
## Cell 11 — Cleaning Step 4: Remove PII patterns

**Why:** ToxicChat and HH-RLHF contain real user conversations that may include email addresses, phone numbers, or names. Leaving PII in training data is an ethical and data protection risk — especially since your platform targets minors.

In [ ]:
def remove_pii(text):
    """Remove common PII patterns from text."""
    if not isinstance(text, str):
        return text
    # Email addresses
    text = re.sub(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', '[EMAIL]', text)
    # Phone numbers (various formats)
    text = re.sub(r'\+?\d[\d\s\-\.\(\)]{7,}\d', '[PHONE]', text)
    # URLs
    text = re.sub(r'https?://\S+', '[URL]', text)
    text = re.sub(r'www\.\S+', '[URL]', text)
    # Social security / ID-like numbers (e.g. 123-45-6789)
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[ID]', text)
    # Collapse multiple spaces
    text = re.sub(r'  +', ' ', text).strip()
    return text

tqdm.pandas(desc="Removing PII")
df['text'] = df['text'].progress_apply(remove_pii)
print(f"PII removal complete. Shape: {df.shape}")

# Spot check
print("\nSample after PII removal:")
print(df['text'].iloc[100])

---
## Cell 12 — Cleaning Step 5: SRH relevance boost

**Why:** Your classifier will be deployed specifically in an SRH chatbot. Generic safety datasets contain many off-topic harmful examples (e.g. bomb-making, financial fraud). We **upweight SRH-relevant rows** rather than remove off-topic ones — we want the classifier to handle both SRH-specific and general harmful inputs, but we want the training data to have proportionally more SRH context.

This cell adds an `is_srh_relevant` flag so you can either (a) oversample SRH rows or (b) use as a feature.

In [ ]:
# SRH-relevant keyword list — covers your 7 topic classes
SRH_KEYWORDS = [
    # Contraception
    'contracepti', 'condom', 'birth control', 'pill', 'iud', 'implant',
    'family planning', 'plan b', 'emergency contracepti',
    # STIs/HIV
    'sti', 'std', 'hiv', 'aids', 'chlamydia', 'gonorrhea', 'syphilis',
    'herpes', 'hpv', 'sexually transmitted',
    # Pregnancy
    'pregnant', 'pregnancy', 'miscarriage', 'abortion', 'prenatal',
    'antenatal', 'fetal', 'trimester', 'conception',
    # Puberty / body
    'puberty', 'menstruat', 'period', 'menarche', 'erection',
    'ejaculat', 'breast', 'reproductive',
    # Consent / GBV
    'consent', 'rape', 'sexual assault', 'gbv', 'gender.based violence',
    'abuse', 'coercion', 'exploitation', 'trafficking',
    # Sexuality / relationships
    'sexual health', 'sex education', 'lgbtq', 'homosexual', 'transgender',
    'intimate partner',
    # Disability + SRH
    'disability', 'pwd', 'accessible',
]

def is_srh_relevant(text):
    t = text.lower()
    return any(kw in t for kw in SRH_KEYWORDS)

tqdm.pandas(desc="Flagging SRH relevance")
df['is_srh'] = df['text'].progress_apply(is_srh_relevant)

print(f"SRH-relevant rows: {df['is_srh'].sum():,} / {len(df):,} ({df['is_srh'].mean()*100:.1f}%)")
print(f"\nSRH relevance by label:")
print(df.groupby('label')['is_srh'].sum())

---
## Cell 13 — Cleaning Step 6: Text normalisation

**Why:** Inconsistent encoding, unusual whitespace, and HTML artefacts can confuse tokenizers.

In [ ]:
def normalise_text(text):
    if not isinstance(text, str):
        return str(text)
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Replace non-breaking spaces and other unicode spaces
    text = re.sub(r'[\u00a0\u200b\u202f\ufeff]', ' ', text)
    # Collapse newlines to space
    text = re.sub(r'[\n\r\t]+', ' ', text)
    # Collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text)
    # Strip leading/trailing
    return text.strip()

tqdm.pandas(desc="Normalising text")
df['text'] = df['text'].progress_apply(normalise_text)

# Final shape check
print(f"After normalisation: {df.shape}")
print(f"Label distribution:")
print(df['label'].value_counts())

---
## Cell 14 — Save clean (pre-balanced) dataset

In [ ]:
# Save the full cleaned (but not yet balanced) dataset
# This is useful for inspection — you can review it before the final sampling step
df.to_csv(f'{OUTPUT_DIR}/safety_classifier_clean.csv', index=False)
print(f"Saved: {OUTPUT_DIR}/safety_classifier_clean.csv")
print(f"Shape: {df.shape}")
print(f"\nClass balance:")
print(df['label'].value_counts())
print(f"\nSource breakdown:")
print(df.groupby(['source', 'label']).size().unstack(fill_value=0))

---
## Cell 15 — Balancing: stratified sampling

**Why:** The raw data is heavily imbalanced (UNSAFE >> SAFE from BeaverTails). An imbalanced classifier will learn to predict UNSAFE for everything and achieve ~80% accuracy while being useless. We balance to 50/50.

**Strategy:** 
- Prioritise SRH-relevant rows in both classes (they match your deployment domain)
- Sample `TARGET_PER_CLASS` from each label
- Within each label, oversample SRH rows proportionally

In [ ]:
def sample_class(df_class, n, seed=RANDOM_SEED):
    """Sample n rows from a class, prioritising SRH-relevant rows."""
    srh_rows    = df_class[df_class['is_srh'] == True]
    nonsrh_rows = df_class[df_class['is_srh'] == False]

    # Aim for 40% SRH-relevant in the sample (floor)
    srh_target    = min(int(n * 0.4), len(srh_rows))
    nonsrh_target = n - srh_target

    srh_sample    = srh_rows.sample(srh_target, random_state=seed)

    # If not enough non-SRH rows, fill with more SRH rows
    if len(nonsrh_rows) < nonsrh_target:
        extra        = nonsrh_target - len(nonsrh_rows)
        leftover_srh = srh_rows.drop(srh_sample.index)
        extra_sample = leftover_srh.sample(min(extra, len(leftover_srh)), random_state=seed)
        nonsrh_sample = pd.concat([nonsrh_rows, extra_sample])
    else:
        nonsrh_sample = nonsrh_rows.sample(nonsrh_target, random_state=seed)

    return pd.concat([srh_sample, nonsrh_sample])


safe_pool   = df[df['label'] == 0]
unsafe_pool = df[df['label'] == 1]

print(f"SAFE pool:   {len(safe_pool):,} rows ({safe_pool['is_srh'].sum():,} SRH)")
print(f"UNSAFE pool: {len(unsafe_pool):,} rows ({unsafe_pool['is_srh'].sum():,} SRH)")

safe_sample   = sample_class(safe_pool,   TARGET_PER_CLASS)
unsafe_sample = sample_class(unsafe_pool, TARGET_PER_CLASS)

balanced = pd.concat([safe_sample, unsafe_sample], ignore_index=True)
balanced = balanced.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"\nBalanced dataset shape: {balanced.shape}")
print(f"SAFE (0):   {(balanced['label']==0).sum():,}")
print(f"UNSAFE (1): {(balanced['label']==1).sum():,}")
print(f"SRH rows:   {balanced['is_srh'].sum():,} ({balanced['is_srh'].mean()*100:.1f}%)")

---
## Cell 16 — Train / Validation / Test split

**Why:** Standard ML practice. We use 70/15/15 rather than 80/10/10 because safety classification is a high-stakes task — we want a larger validation set for hyperparameter tuning.

In [ ]:
# Stratified split to maintain 50/50 class balance in each split
X = balanced['text']
y = balanced['label']
meta = balanced[['source', 'is_srh']]

X_trainval, X_test, y_trainval, y_test, meta_trainval, meta_test = train_test_split(
    X, y, meta, test_size=0.15, stratify=y, random_state=RANDOM_SEED
)
X_train, X_val, y_train, y_val, meta_train, meta_val = train_test_split(
    X_trainval, y_trainval, meta_trainval,
    test_size=0.15/0.85,   # 15% of original = 15/85 of the trainval
    stratify=y_trainval,
    random_state=RANDOM_SEED
)

def make_split_df(X, y, meta, split_name):
    d = pd.concat([X, y, meta], axis=1)
    d['split'] = split_name
    return d

train_df = make_split_df(X_train, y_train, meta_train, 'train')
val_df   = make_split_df(X_val,   y_val,   meta_val,   'val')
test_df  = make_split_df(X_test,  y_test,  meta_test,  'test')

print(f"Train: {len(train_df):,} rows  | SAFE={y_train.sum() - y_train.sum() + (y_train==0).sum()} | UNSAFE={(y_train==1).sum()}")
print(f"Val:   {len(val_df):,} rows   | SAFE={(y_val==0).sum()} | UNSAFE={(y_val==1).sum()}")
print(f"Test:  {len(test_df):,} rows  | SAFE={(y_test==0).sum()} | UNSAFE={(y_test==1).sum()}")

---
## Cell 17 — Save final dataset files

In [ ]:
# Save the full final balanced dataset
final_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
final_df.to_csv(f'{OUTPUT_DIR}/safety_classifier_final.csv', index=False)

# Save individual splits
train_df.to_csv(f'{OUTPUT_DIR}/safety_train.csv', index=False)
val_df.to_csv(f'{OUTPUT_DIR}/safety_val.csv',     index=False)
test_df.to_csv(f'{OUTPUT_DIR}/safety_test.csv',   index=False)

print("Saved files:")
for f in ['safety_classifier_raw.csv','safety_classifier_clean.csv',
          'safety_classifier_final.csv','safety_train.csv',
          'safety_val.csv','safety_test.csv']:
    path = f'{OUTPUT_DIR}/{f}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        rows    = pd.read_csv(path).shape[0]
        print(f"  {f:45s} {rows:>6,} rows  {size_mb:.1f} MB")

---
## Cell 18 — EDA visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Safety Classifier Dataset — Final EDA', fontsize=14, fontweight='bold')

# 1. Class balance
final_df['label'].map({0:'SAFE', 1:'UNSAFE'}).value_counts().plot(
    kind='bar', ax=axes[0,0], color=['#10B981','#EF4444'], edgecolor='white'
)
axes[0,0].set_title(f'Class balance (n={len(final_df):,})')
axes[0,0].tick_params(axis='x', rotation=0)
axes[0,0].set_ylabel('Count')

# 2. Source breakdown
final_df.groupby(['source','label']).size().unstack().plot(
    kind='barh', ax=axes[0,1], color=['#10B981','#EF4444']
)
axes[0,1].set_title('Source breakdown by class')
axes[0,1].legend(['SAFE','UNSAFE'])

# 3. Word length distribution by class
final_df['wlen'] = final_df['text'].str.split().str.len()
for label, color, name in [(0,'#10B981','SAFE'),(1,'#EF4444','UNSAFE')]:
    final_df[final_df['label']==label]['wlen'].hist(
        bins=40, ax=axes[1,0], alpha=0.6, color=color, label=name
    )
axes[1,0].set_title('Word count by class')
axes[1,0].set_xlabel('Words')
axes[1,0].legend()

# 4. SRH relevance by split
srh_by_split = final_df.groupby('split')['is_srh'].mean() * 100
srh_by_split.plot(kind='bar', ax=axes[1,1], color='steelblue', edgecolor='white')
axes[1,1].set_title('SRH-relevant rows (%) by split')
axes[1,1].set_ylabel('% SRH relevant')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_safety_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/fig_safety_eda.png")

---
## Cell 19 — Final summary and schema documentation

In [ ]:
print('=' * 65)
print('SAFETY CLASSIFIER DATASET — FINAL SUMMARY')
print('=' * 65)
print(f'Total rows:           {len(final_df):,}')
print(f'  SAFE   (label=0):   {(final_df["label"]==0).sum():,}')
print(f'  UNSAFE (label=1):   {(final_df["label"]==1).sum():,}')
print(f'Train:                {len(train_df):,}')
print(f'Validation:           {len(val_df):,}')
print(f'Test:                 {len(test_df):,}')
print(f'SRH-relevant rows:    {final_df["is_srh"].sum():,} ({final_df["is_srh"].mean()*100:.1f}%)')
print()
print('Column schema:')
print('  text      str   → User query text (cleaned, PII-removed)')
print('  label     int   → 0=SAFE, 1=UNSAFE')
print('  source    str   → Original dataset name')
print('  is_srh    bool  → SRH keyword match flag')
print('  split     str   → train / val / test')
print()
print('Sources:')
print(final_df['source'].value_counts().to_string())
print()
print('Output files:')
print(f'  data/safety/safety_classifier_final.csv  ← USE THIS for training')
print(f'  data/safety/safety_train.csv             ← Train split')
print(f'  data/safety/safety_val.csv               ← Validation split')
print(f'  data/safety/safety_test.csv              ← Test split (hold out!)')
print(f'  data/safety/safety_classifier_clean.csv  ← Full cleaned pre-balance')
print(f'  data/safety/safety_classifier_raw.csv    ← Raw merged (reference)')
print('=' * 65)